In [218]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [219]:
import sys
print(sys.executable)

d:\LLM\trust-behaviours\.venv\Scripts\python.exe


In [220]:
from dotenv import load_dotenv
load_dotenv()

import kagglehub
import pandas as pd
import os
from torch.utils.data import Dataset, DataLoader, random_split
from torch import nn, optim
import torch
from tqdm import tqdm

EMBED_DIM = 128
HIDDEN_DIM = 64
MAX_EPOCHS = 15

VOCAB_SIZE = 20000
MIN_OCCURRENCE = 23

path = kagglehub.dataset_download("snap/amazon-fine-food-reviews")
df = pd.read_csv(os.path.join(path, "Reviews.csv"), usecols=["Text", "Score"])

In [221]:
df["Text"] = df["Text"].str.lower()
df["Text"] = df["Text"].str.replace(r"<[^>]+>", " ", regex=True)
df["Text"] = df["Text"].str.replace(r"[^\w\s]", " ", regex=True)

In [222]:
# Build vocabulary
from collections import Counter
word_freq = Counter(" ".join(df["Text"]).split())
most_common = word_freq.most_common(VOCAB_SIZE)

# Tokens spéciaux
tokens_list = {"<PAD>": 0, "<UNK>": 1}

for word, _ in most_common:
    tokens_list[word] = len(tokens_list)

In [223]:
def tokenize(text, max_length=222):
    sentence = [tokens_list.get(word, tokens_list["<UNK>"]) for word in text.split()]
    # Pad or truncate the sentence to the maximum length
    if len(sentence) < max_length:
        sentence.extend([tokens_list["<PAD>"]] * (max_length - len(sentence)))
    else:
        sentence = sentence[:max_length]
    return sentence

df["tokens"] = df["Text"].apply(tokenize)

In [224]:
class ReviewsDataset(Dataset):
    def __init__(self, df):
        self.sequences = torch.tensor(df["tokens"].tolist(), dtype=torch.long)
        self.labels = torch.tensor(df["Score"].tolist(), dtype=torch.long) - 1  # Scores de 1 à 5 -> labels de 0 à 4

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return (
            self.sequences[idx], 
            self.labels[idx]
        )

In [225]:
class FeelingModel(nn.Module):
    def __init__(self, vocab_size, embed_dim = 256, hidden_dim = 128, output_dim = 5):
        super(FeelingModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=2, bidirectional=True, dropout=0.3, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        output = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(output)

In [226]:
train_size = int(0.7 * len(df))
val_size = int(0.15 * len(df))
test_size = len(df) - train_size - val_size

dataset = ReviewsDataset(df)
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size], 
                                                        generator=torch.Generator().manual_seed(42))

In [227]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

In [228]:
device = torch.device("cpu")
if torch.cuda.is_available():
  device = torch.device("cuda")
elif torch.backends.mps.is_available():
  device = torch.device("mps")

print(device)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA device")

model = FeelingModel(vocab_size=len(tokens_list), embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

cuda
NVIDIA GeForce RTX 4060 Ti


In [229]:
previous_val_loss = float('inf')
for epoch in range(MAX_EPOCHS):
    # Entrainement
    model.train()
    for X_batch, Y_batch in tqdm(train_loader, desc=f"Epoch {epoch+1} - Training"):
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, Y_batch)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    with torch.no_grad():
        for X_batch, Y_batch in tqdm(val_loader, desc=f"Epoch {epoch+1} - Validation"):
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)
            output = model(X_batch)
            val_loss_batch = criterion(output, Y_batch)   # ← calcul réel
            predicted_value = output.argmax(dim=1)
            correct += (Y_batch == predicted_value).sum().item()
            total += Y_batch.size(0)
            val_loss += val_loss_batch.item()
    val_accuracy = correct / total
    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1}/{MAX_EPOCHS} | val_loss: {val_loss:.4f} | val_accuracy: {val_accuracy:.4f} | lr: {optimizer.param_groups[0]['lr']:.6f}")
    scheduler.step(val_loss)
    
    if (previous_val_loss - val_loss) < 0.003:
        break
    previous_val_loss = val_loss

Epoch 1 - Validation: 100%|██████████| 1333/1333 [00:03<00:00, 394.52it/s]


Epoch 1/15 | val_loss: 0.8938 | val_accuracy: 0.6766 | lr: 0.000100


Epoch 2 - Validation: 100%|██████████| 1333/1333 [00:03<00:00, 405.96it/s]


Epoch 2/15 | val_loss: 0.7942 | val_accuracy: 0.6989 | lr: 0.000100


Epoch 3 - Validation: 100%|██████████| 1333/1333 [00:03<00:00, 406.78it/s]


...

Epoch 10/15 | val_loss: 0.6243 | val_accuracy: 0.7637 | lr: 0.000100


Epoch 11 - Validation: 100%|██████████| 1333/1333 [00:03<00:00, 407.01it/s]


Epoch 11/15 | val_loss: 0.6155 | val_accuracy: 0.7708 | lr: 0.000100


Epoch 12 - Validation: 100%|██████████| 1333/1333 [00:03<00:00, 406.66it/s]

Epoch 12/15 | val_loss: 0.6136 | val_accuracy: 0.7729 | lr: 0.000100


In [230]:
torch.save(model.state_dict(), "../model.pt")

In [231]:
# C'est parti pour le test !
model.load_state_dict(torch.load("../model.pt"))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for X_batch, Y_batch in tqdm(test_loader, desc="Testing"):
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)
        output = model(X_batch)
        predicted_value = output.argmax(dim=1)
        all_preds.extend(predicted_value.cpu().numpy())
        all_labels.extend(Y_batch.cpu().numpy())

from sklearn.metrics import classification_report
print(classification_report(all_labels, all_preds, target_names=["1","2","3","4","5"]))

Testing:  16%|█▌        | 210/1333 [00:00<00:02, 417.66it/s]

Testing: 100%|██████████| 1333/1333 [00:03<00:00, 417.70it/s]

              precision    recall  f1-score   support

           1       0.71      0.74      0.72      7788
           2       0.45      0.28      0.35      4391
           3       0.48      0.45      0.47      6398
           4       0.53      0.38      0.44     12089
           5       0.86      0.94      0.90     54603

    accuracy                           0.77     85269
   macro avg       0.61      0.56      0.58     85269
weighted avg       0.75      0.77      0.76     85269

